### Column Descriptions

- **`stock_id`**: A unique identifier for the stock. Not all stock IDs exist in every time bucket.
- **`date_id`**: A unique identifier for the date. Date IDs are sequential and consistent across all stocks.
- **`seconds_in_bucket`**: The number of seconds elapsed since the beginning of the day's closing auction, always starting from 0.
- **`imbalance_size`**: The amount unmatched at the current reference price (in USD).
- **`imbalance_buy_sell_flag`**: An indicator reflecting the direction of auction imbalance:
  - `1`: Buy-side imbalance.
  - `-1`: Sell-side imbalance.
  - `0`: No imbalance.
- **`reference_price`**: The price at which paired shares are maximized, the imbalance is minimized, and the distance from the bid-ask midpoint is minimized (in that order). It can also be thought of as the near price bounded between the best bid and ask price.
- **`matched_size`**: The amount that can be matched at the current reference price (in USD).
- **`far_price`**: The crossing price that will maximize the number of shares matched based on auction interest only (excludes continuous market orders).
- **`near_price`**: The crossing price that will maximize the number of shares matched based on auction and continuous market orders.
- **`bid_price`**: Price of the most competitive buy level in the non-auction book.
- **`bid_size`**: The dollar notional amount on the most competitive buy level in the non-auction book.
- **`ask_price`**: Price of the most competitive sell level in the non-auction book.
- **`ask_size`**: The dollar notional amount on the most competitive sell level in the non-auction book.
- **`wap`**: The weighted average price in the non-auction book. Calculated as: 
$$
\text{wap} = \frac{\text{BidPrice} \times \text{AskSize} + \text{AskPrice} \times \text{BidSize}}{\text{BidSize} + \text{AskSize}}
$$
- **`target`**: The 60-second future move in the WAP of the stock, less the 60-second future move of the synthetic index. Only provided for the train set. The unit is basis points (1 basis point = 0.01% price move). The target is calculated as:
$$
\text{Target} = \left( \frac{\text{StockWAP}_{t+60}}{\text{StockWAP}_t} - \frac{\text{IndexWAP}_{t+60}}{\text{IndexWAP}_t} \right) \times 10000
$$

- The synthetic index is a custom weighted index of Nasdaq-listed stocks constructed by Optiver for this competition.
- **`time_id`**: A unique identifier for the time bucket.
- **`row_id`**: A unique identifier for the row.

### Notes:
- All **size-related columns** are in USD terms.
- All **price-related columns** are converted to a price move relative to the stock WAP (weighted average price) at the beginning of the auction period.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession

In [ ]:
sns.set_style("dark")
plt.style.use("dark_background")

In [ ]:
spark = (
    SparkSession.builder.appName("eda")
    .config("spark.driver.memory", "8g")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.2.0,com.amazonaws:aws-java-sdk-bundle:1.11.375")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.access.key", "test")
    .config("spark.hadoop.fs.s3a.secret.key", "test")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)


In [ ]:
# S3 path format
s3_path = "s3a://data/train.csv"

# Set Hadoop configurations for S3 access
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", "test")
hadoop_conf.set("fs.s3a.secret.key", "test")
hadoop_conf.set("fs.s3a.endpoint", "http://localhost:4566")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


# Read the file directly into a PySpark DataFrame
df_spark = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(s3_path)

# Show the first few rows of the DataFrame
df_spark.show(5)

In [ ]:
# S3 path format
# s3_path = "s3://data/train.csv"

# # Read the file directly into a pandas DataFrame
# df = pd.read_csv(s3_path, storage_options={
#     'key': 'test',
#     'secret': 'test',
#     'client_kwargs': {
#         'endpoint_url': 'http://localhost:4566'
#     }
# })

# df.head()

In [ ]:
# Register the DataFrame as a temporary view
df_spark.createOrReplaceTempView("train")

# Now you can query it with SQL
spark.sql(
    """ --sql
        SELECT * 
        FROM train LIMIT 1;
    """
).show()

In [ ]:
spark.sql("show databases").show()
spark.sql("show tables").show()

In [ ]:
print(spark.sparkContext.uiWebUrl)

In [ ]:
spark.sql(
    """ --sql
        SELECT
            stock_id,
            avg(reference_price) as avg_reference_price,
            min(reference_price) as min_reference_price,
            max(reference_price) as max_reference_price
        FROM train
        GROUP BY stock_id
        ORDER BY stock_id ASC
        LIMIT 10;
    """
).toPandas().plot(
    kind="bar", x="stock_id", y=["avg_reference_price", "min_reference_price", "max_reference_price"]
)
plt.show()

In [ ]:
spark.sql(
    """ --sql
        SELECT count(*)
        FROM (
            SELECT
                stock_id,
                count(stock_id) as count_stock_id
            FROM train
            GROUP BY stock_id
            ORDER BY count_stock_id DESC
        )
        WHERE count_stock_id = 26455;
    """
).show()

In [ ]:
spark.sql(
    """ --sql
        SELECT COUNT(DISTINCT stock_id) AS unique_stock_count FROM train;
    """
).show()

In [ ]:
# spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
df = df_spark.toPandas()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print(df.date_id.nunique())
print(df.row_id.nunique())
print(df.time_id.nunique())

In [ ]:
# Convert specific columns to category
df["stock_id"] = df["stock_id"].astype("category")
df["imbalance_buy_sell_flag"] = df["imbalance_buy_sell_flag"].astype("category")
df["date_id"] = df["date_id"].astype("category")
df["time_id"] = df["time_id"].astype("category")

# Splitting the columns
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()

# Display the lists of columns
print("Categorical Columns:", categorical_columns)
print("Numerical Columns:", numerical_columns)

In [ ]:
df[numerical_columns].describe()

In [ ]:
df[categorical_columns].describe()

In [ ]:
# Plot boxplots for each numerical column
for col in numerical_columns:
    plt.figure(figsize=(14, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
(
    df
    .query('stock_id ==0 & date_id ==0')
    [['seconds_in_bucket','bid_price','ask_price', 'wap']]
    .replace(0, np.nan)
    .set_index('seconds_in_bucket')
    .plot(title='Stock 0 on Day 0 - How the order book pricing changes during the auction')
)